# K-Means Clustering on Sales Data
**Goal:** Cluster sales records and determine the optimal number of clusters using the **Elbow Method**.

In [ ]:
# Import essential libraries for data analysis and visualization
import pandas as pd           # Data manipulation and analysis
import numpy as np            # Numerical operations
import matplotlib.pyplot as plt  # Plotting graphs
import seaborn as sns         # Statistical data visualization

## STEP 1 – Load Dataset

In [ ]:
# Load sales dataset with special encoding for unicode characters
# encoding='unicode_escape' handles special characters in address fields
df = pd.read_csv('sales_data_sample.csv', encoding='unicode_escape')

# Display first 5 rows to understand the structure
df.head()

In [ ]:
# Display comprehensive information about the dataset
# Shows: column names, data types, non-null counts, memory usage
df.info()

## STEP 2 – Drop Irrelevant Columns
Remove address, contact, and identifier columns that add no clustering signal.

In [ ]:
# Remove columns that don't contribute to clustering analysis
# These are mostly identifiers and contact information
df_drop = [
    'ADDRESSLINE1', 'ADDRESSLINE2', 'POSTALCODE', 'CITY',
    'TERRITORY', 'PHONE', 'STATE', 'CONTACTFIRSTNAME',
    'CONTACTLASTNAME', 'CUSTOMERNAME', 'ORDERNUMBER'
]

# Drop the specified columns along axis 1 (columns)
df = df.drop(df_drop, axis=1)

# Verify the updated structure
df.info()

## STEP 3 – Explore Remaining Columns

In [ ]:
# Display value counts for each column to understand data distribution
# This helps identify categorical vs numerical features
for col in df.columns.values:
    print(df[col].value_counts())

## STEP 4 – Drop Time-Based Columns
`ORDERDATE`, `STATUS`, `MONTH_ID`, `QTR_ID`, `YEAR_ID` are not useful for product/sales clustering.

In [ ]:
# Remove temporal columns as we're clustering based on sales patterns, not time
# inplace=True modifies the DataFrame directly without creating a copy
df.drop(columns=['ORDERDATE', 'STATUS', 'MONTH_ID', 'QTR_ID', 'YEAR_ID'], inplace=True)

# Display updated DataFrame structure
df.head()

## STEP 5 – Encode Categorical Columns
K-Means requires numeric input — encode `PRODUCTLINE`, `PRODUCTCODE`, `COUNTRY`, `DEALSIZE` using `LabelEncoder`.

In [ ]:
# Import LabelEncoder to convert categorical text into numerical labels
from sklearn.preprocessing import LabelEncoder

def convert_categories(col):
    """Convert categorical column to numeric using LabelEncoder"""
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].values)

# Apply encoding to all categorical columns
categories = ['PRODUCTLINE', 'PRODUCTCODE', 'COUNTRY', 'DEALSIZE']
for col in categories:
    convert_categories(col)

# Display encoded data
df.head()

## STEP 6 – Scale the Data
`StandardScaler` ensures all features contribute equally to distance calculations.

In [ ]:
# Import StandardScaler for feature normalization
from sklearn.preprocessing import StandardScaler

# Initialize and apply StandardScaler
sc   = StandardScaler()
data = sc.fit_transform(df)

# Scaling transforms all features to have mean=0 and std=1
# Critical for K-Means as it uses Euclidean distance

## STEP 7 – Elbow Method (Find Optimal K)
For each K, compute **WCSS** (Within-Cluster Sum of Squares = `inertia_`).  
The "elbow" point — where WCSS stops dropping sharply — is the optimal K.

In [ ]:
# Import K-Means clustering algorithm
from sklearn.cluster import KMeans

# Test different values of K (number of clusters) from 2 to 10
wcss = []
for i in range(2, 11):
    # Create K-Means model with i clusters
    # n_init=10 runs algorithm 10 times with different initializations
    km = KMeans(n_clusters=i, init='k-means++', n_init=10, random_state=42)
    km.fit(data)  # Fit the model to scaled data
    wcss.append(km.inertia_)  # Store the WCSS (inertia)

# Plot the Elbow curve
plt.figure(figsize=(10, 6))
plt.plot(range(2, 11), wcss, marker='o', linewidth=2, markersize=8)
plt.title('Elbow Method For Optimal K', fontsize=16, weight='bold')
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('WCSS (Within-Cluster Sum of Squares)', fontsize=12)
plt.grid(alpha=0.3)
plt.show()

# The "elbow" is where the curve bends significantly
# This indicates diminishing returns from adding more clusters

## STEP 8 – Apply K-Means with Optimal K

In [ ]:
# Based on elbow curve, choose optimal K (typically where curve bends)
# Here we use K=3 as an example
optimal_k = 3

# Create and fit final K-Means model
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', n_init=10, random_state=42)
clusters = kmeans.fit_predict(data)

# Add cluster labels to original DataFrame
df['Cluster'] = clusters

print(f'Data successfully clustered into {optimal_k} groups')
print(f'\nCluster distribution:')
print(df['Cluster'].value_counts().sort_index())

## STEP 9 – Visualize Clusters

In [ ]:
# For 2D visualization, we'll use two important features
# In real scenarios, use PCA for dimensionality reduction with many features

plt.figure(figsize=(12, 6))

# Scatter plot of data points colored by cluster
for i in range(optimal_k):
    cluster_data = df[df['Cluster'] == i]
    plt.scatter(
        cluster_data['QUANTITYORDERED'], 
        cluster_data['SALES'],
        label=f'Cluster {i}',
        alpha=0.6,
        s=50
    )

# Plot cluster centers (need to inverse transform scaled centers)
centers = sc.inverse_transform(kmeans.cluster_centers_)
plt.scatter(
    centers[:, 0],  # QUANTITYORDERED index
    centers[:, 3],  # SALES index
    marker='X',
    s=300,
    c='red',
    edgecolors='black',
    linewidths=2,
    label='Centroids'
)

plt.title('K-Means Clustering: Quantity vs Sales', fontsize=14, weight='bold')
plt.xlabel('Quantity Ordered', fontsize=12)
plt.ylabel('Sales ($)', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## STEP 10 – Analyze Cluster Characteristics

In [ ]:
# Analyze characteristics of each cluster
# This helps understand what each cluster represents

print('\n=== Cluster Analysis ===')
for i in range(optimal_k):
    cluster_data = df[df['Cluster'] == i]
    print(f'\n--- Cluster {i} ---')
    print(f'Size: {len(cluster_data)} records')
    print(f'Average Quantity Ordered: {cluster_data["QUANTITYORDERED"].mean():.2f}')
    print(f'Average Sales: ${cluster_data["SALES"].mean():.2f}')
    print(f'Average Price Each: ${cluster_data["PRICEEACH"].mean():.2f}')
    print(f'Most common Product Line: {cluster_data["PRODUCTLINE"].mode()[0] if len(cluster_data) > 0 else "N/A"}')
    print(f'Most common Deal Size: {cluster_data["DEALSIZE"].mode()[0] if len(cluster_data) > 0 else "N/A"}')

---
# 📚 Practical Examination Explanation

## 🔧 Libraries Used

| Library | Purpose |
|---------|----------|
| **pandas** | Loading CSV data, DataFrame manipulation, data exploration |
| **numpy** | Numerical operations, array handling |
| **matplotlib.pyplot** | Creating visualizations (elbow curve, scatter plots) |
| **seaborn** | Enhanced statistical visualizations |
| **sklearn.cluster** | K-Means clustering algorithm implementation |
| **sklearn.preprocessing** | LabelEncoder for categorical encoding, StandardScaler for normalization |

---

## 📖 Technical Terms Explained

### 1. **K-Means Clustering**
- **Unsupervised learning algorithm** that groups similar data points into K clusters
- **Goal**: Minimize within-cluster variance (WCSS)
- **Centroid-based**: Each cluster has a center point (centroid)
- **Iterative process**: Points are repeatedly reassigned to nearest centroid

### 2. **WCSS (Within-Cluster Sum of Squares)**
```python
WCSS = Σ (distance from point to its centroid)²
```
- Also called **inertia** in sklearn
- **Lower WCSS** = tighter, more compact clusters
- Used to evaluate cluster quality

### 3. **Elbow Method**
- **Technique** for finding optimal number of clusters (K)
- **Process**: Plot WCSS vs K values
- **Elbow point**: Where WCSS starts decreasing slowly
- **Visual analogy**: Graph looks like a bent arm, the "elbow" is optimal K

### 4. **Centroid**
- **Definition**: Center point of a cluster
- **Calculated as**: Mean of all points in that cluster
- **Represented by**: `kmeans.cluster_centers_`
- **Updates**: Recalculated each iteration until convergence

### 5. **K-Means++ Initialization**
```python
init='k-means++'
```
- **Smart centroid initialization** method
- **Better than random**: Spreads initial centroids apart
- **Result**: Faster convergence, better final clusters
- **Default** in sklearn

### 6. **Feature Scaling / Normalization**
```python
StandardScaler: X_scaled = (X - μ) / σ
```
- **Critical for K-Means**: Algorithm uses Euclidean distance
- **Without scaling**: Features with large values dominate
- **StandardScaler**: mean = 0, std = 1
- **Example**: Sales ($10,000) vs Quantity (50) - sales would dominate without scaling

### 7. **Label Encoding**
- **Converts** categorical text to numerical labels
- **Example**: ['USA', 'France', 'UK'] → [2, 0, 1]
- **Limitation**: Creates ordinal relationship where none exists
- **Alternative**: One-Hot Encoding (better but creates more features)

### 8. **Unsupervised Learning**
- **No labels**: Algorithm finds patterns without being told what to look for
- **vs Supervised**: No target variable to predict
- **Goal**: Discover hidden structure in data
- **Examples**: Clustering, dimensionality reduction, anomaly detection

---

## ⚙️ K-Means Algorithm

### How K-Means Works (Step-by-Step)

**Initialization:**
1. Choose K (number of clusters)
2. Initialize K centroids (randomly or with k-means++)

**Iteration Loop:**
```
Repeat until convergence:
  1. Assignment Step:
     - For each data point:
       - Calculate distance to all K centroids
       - Assign point to nearest centroid
  
  2. Update Step:
     - For each cluster:
       - Recalculate centroid as mean of assigned points
  
  3. Check Convergence:
     - If centroids don't change (or change very little): STOP
     - Else: Continue to next iteration
```

### Distance Calculation (Euclidean)
```python
distance = √[(x₁-x₂)² + (y₁-y₂)² + ... + (zₙ-z₂)²]
```

### Convergence Criteria
- **Centroids stabilize**: No (or minimal) movement between iterations
- **Max iterations reached**: sklearn default = 300
- **Tolerance threshold**: sklearn default tol=1e-4

### Key Parameters

```python
KMeans(
    n_clusters=3,        # Number of clusters (K)
    init='k-means++',    # Initialization method
    n_init=10,          # Number of times to run with different seeds
    max_iter=300,       # Maximum iterations per run
    random_state=42     # Seed for reproducibility
)
```

- **n_clusters**: The K in K-Means
- **n_init**: Run algorithm multiple times, keep best result (lowest WCSS)
- **max_iter**: Prevent infinite loops
- **random_state**: Makes results reproducible

---

## 🔄 Workflow Summary

```
1. Load Data → sales_data_sample.csv
   ↓
2. Data Cleaning
   - Drop irrelevant columns (addresses, contacts, IDs)
   - Drop temporal columns (dates, quarters, months)
   ↓
3. Feature Engineering
   - Encode categorical variables (LabelEncoder)
   - Convert: PRODUCTLINE, PRODUCTCODE, COUNTRY, DEALSIZE
   ↓
4. Feature Scaling (CRITICAL)
   - StandardScaler: mean=0, std=1
   - Ensures equal contribution from all features
   ↓
5. Determine Optimal K
   - Test K from 2 to 10
   - Plot WCSS vs K (Elbow Method)
   - Identify elbow point
   ↓
6. Apply K-Means
   - Use optimal K
   - fit_predict() to get cluster labels
   ↓
7. Visualization
   - Scatter plot with cluster colors
   - Mark centroids
   - Use 2D features or apply PCA
   ↓
8. Cluster Analysis
   - Calculate statistics per cluster
   - Identify cluster characteristics
   - Business interpretation
```

---

## 🏢 Real-World Applications

### Customer Segmentation
- **Marketing**: Target different customer groups
- **Personalization**: Tailor products/services
- **Example**: High-value, medium-value, low-value customers

### Sales Analysis
- **Product grouping**: Similar sales patterns
- **Regional analysis**: Geographic clusters
- **Pricing strategy**: Cluster-based pricing

### Image Compression
- Reduce color palette
- Cluster similar colors
- Smaller file size

### Anomaly Detection
- Points far from all centroids = outliers
- Fraud detection
- Quality control

### Document Clustering
- Topic modeling
- News article grouping
- Search result organization

### Recommendation Systems
- Group similar items
- Group similar users
- Suggest based on cluster membership

---

## 🎯 Advantages & Limitations

### ✅ Advantages
1. **Simple and intuitive** algorithm
2. **Fast and efficient** for large datasets
3. **Scalable** to many samples
4. **Guaranteed convergence** (to local optimum)
5. **Well-suited for spherical clusters**
6. **Easy to implement** and interpret

### ❌ Limitations
1. **Must specify K beforehand** (use Elbow Method)
2. **Sensitive to initialization** (use k-means++)
3. **Sensitive to outliers** (use preprocessing)
4. **Assumes spherical clusters** (struggles with irregular shapes)
5. **Sensitive to feature scaling** (MUST normalize)
6. **May converge to local optimum** (run multiple times with n_init)
7. **Curse of dimensionality** (performance degrades with many features)

---

## 🚀 Improvement Suggestions

### 1. **Silhouette Score** (Alternative to Elbow)
```python
from sklearn.metrics import silhouette_score

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(data)
    score = silhouette_score(data, labels)
    print(f'K={k}, Silhouette Score: {score:.3f}')
    
# Higher score (closer to 1) = better clustering
```

### 2. **PCA for Visualization**
```python
from sklearn.decomposition import PCA

# Reduce to 2D for visualization
pca = PCA(n_components=2)
data_2d = pca.fit_transform(data)

# Plot clusters in 2D space
plt.scatter(data_2d[:, 0], data_2d[:, 1], c=clusters, cmap='viridis')
```

### 3. **Outlier Removal**
```python
from scipy import stats

# Remove outliers using Z-score
z_scores = np.abs(stats.zscore(df.select_dtypes(include=[np.number])))
df_clean = df[(z_scores < 3).all(axis=1)]
```

### 4. **One-Hot Encoding** (Better for Categorical)
```python
# Instead of LabelEncoder
df_encoded = pd.get_dummies(df, columns=['PRODUCTLINE', 'COUNTRY', 'DEALSIZE'])
```

### 5. **Alternative Clustering Algorithms**
```python
# DBSCAN - handles irregular shapes, finds outliers
from sklearn.cluster import DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=5)

# Hierarchical Clustering - creates dendrogram
from sklearn.cluster import AgglomerativeClustering
hc = AgglomerativeClustering(n_clusters=3)
```

### 6. **Davies-Bouldin Index**
```python
from sklearn.metrics import davies_bouldin_score

# Lower score = better clustering
score = davies_bouldin_score(data, clusters)
```

### 7. **Feature Importance Analysis**
```python
# Analyze which features separate clusters best
for i in range(optimal_k):
    cluster_data = df[df['Cluster'] == i]
    print(f'\nCluster {i} Characteristics:')
    print(cluster_data.describe())
```

---

## 📝 Exam Preparation Tips

### Key Concepts to Remember

1. **Why scaling is mandatory for K-Means?**
   - K-Means uses Euclidean distance
   - Features with larger scales dominate distance calculations
   - Example: Sales ($10,000) vs Quantity (50) - sales dominates without scaling

2. **What is the Elbow Method?**
   - Visual technique to find optimal K
   - Plot WCSS vs K
   - Elbow point = where adding more clusters gives diminishing returns
   - Not always clear - use Silhouette Score as backup

3. **WCSS vs Inertia**
   - Same thing, different names
   - WCSS = academic term
   - Inertia = sklearn's attribute name (`kmeans.inertia_`)

4. **K-Means++ vs Random Initialization**
   - **K-Means++**: Smart initial centroid placement
   - **Random**: May get stuck in poor local optima
   - **Always use k-means++** (sklearn default)

5. **Supervised vs Unsupervised Learning**
   - **Supervised**: Has labels/target variable (classification, regression)
   - **Unsupervised**: No labels (clustering, dimensionality reduction)
   - **K-Means**: Unsupervised

6. **When K-Means fails?**
   - Non-spherical clusters (use DBSCAN)
   - Varying cluster sizes
   - Varying densities
   - Many outliers

### Common Exam Questions

**Q1: Why do we drop time-based columns?**
- Focus on product/sales patterns, not temporal trends
- Temporal features create clusters based on time, not behavior
- For time-series clustering, use different approaches

**Q2: LabelEncoder vs One-Hot Encoding?**
- **LabelEncoder**: Creates ordinal relationship (USA=2, France=0 implies order)
- **One-Hot**: No artificial ordering, but increases dimensions
- **For K-Means**: One-Hot is better but creates more features

**Q3: What if there's no clear elbow?**
- Use Silhouette Score (choose K with highest score)
- Use Davies-Bouldin Index (choose K with lowest score)
- Consider business requirements (e.g., "we want 3-5 customer segments")

**Q4: How to interpret cluster results?**
- Calculate cluster statistics (mean, median per feature)
- Identify distinguishing characteristics
- Give business-meaningful names (e.g., "High-value customers")

**Q5: n_init parameter purpose?**
- Runs algorithm multiple times with different random initializations
- Keeps best result (lowest WCSS)
- Helps avoid poor local optima
- Default n_init=10 in sklearn

**Q6: Can K-Means handle missing values?**
- No, must impute missing values first
- Options: mean, median, mode imputation
- Or remove rows with missing values

### Code Snippets to Remember

```python
# Essential imports
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Basic K-Means workflow
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

kmeans = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)
clusters = kmeans.fit_predict(data_scaled)

# Elbow Method
wcss = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, init='k-means++', n_init=10)
    km.fit(data_scaled)
    wcss.append(km.inertia_)
    
# Access results
centroids = kmeans.cluster_centers_
inertia = kmeans.inertia_
labels = kmeans.labels_
```

### Visualization Remember Points

- **Elbow curve**: X-axis = K, Y-axis = WCSS
- **Cluster plot**: Different colors for each cluster
- **Centroids**: Marked with special symbol (X, star, etc.)
- **2D visualization**: Use PCA if more than 2 features

---

## 🎓 Summary

This practical demonstrates:
- ✅ **Unsupervised learning** with K-Means clustering
- ✅ **Data preprocessing** (encoding, scaling, feature selection)
- ✅ **Elbow Method** for determining optimal K
- ✅ **Feature scaling** importance for distance-based algorithms
- ✅ **Cluster visualization** and interpretation
- ✅ **Business insights** from cluster analysis
- ✅ **Sales data segmentation** for strategic decisions

**Key Takeaway:** K-Means is a powerful, simple algorithm for discovering natural groupings in data. Critical steps are feature scaling and choosing optimal K using the Elbow Method. Always interpret clusters in business context to derive actionable insights.

---